# Download de Documentos — Levantamento de Políticas de IA em Universidades Brasileiras

Este notebook lê o arquivo CSV gerado no levantamento e tenta baixar automaticamente cada documento disponível.

**O que ele faz:**
- Lê o CSV com os metadados de cada documento
- Filtra apenas os registros com URL disponível
- Tenta fazer o download de cada arquivo
- Organiza os arquivos em subpastas por região
- Gera um relatório final com o que foi baixado, o que falhou e o que precisa de atenção manual

**Pré-requisitos:**
```
pip install requests pandas tqdm
```

> ⚠️ Coloque o arquivo `levantamento_ia_universidades_brasileiras.csv` na mesma pasta deste notebook antes de executar.

## 1. Imports e configurações gerais

Importamos as bibliotecas necessárias e definimos as configurações do script: pasta de destino, headers HTTP (para que os servidores não rejeitem a requisição como bot) e timeout por download.

In [ ]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from urllib.parse import urlparse

# Pasta raiz onde os documentos serão salvos
PASTA_DESTINO = Path("documentos_ia_universidades")

# Tempo máximo de espera por download (segundos)
TIMEOUT = 30

# Intervalo entre requisições (segundos) — evita sobrecarregar os servidores
INTERVALO = 1.5

# Headers que simulam um navegador comum
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/pdf,text/html,application/xhtml+xml,*/*",
}

print("✅ Configurações carregadas.")

## 2. Leitura e pré-visualização do CSV

Carregamos o CSV do levantamento e exibimos um resumo. Filtramos apenas os registros que possuem URL — os sem link precisarão de busca manual.

In [ ]:
df = pd.read_csv("levantamento_ia_universidades_brasileiras.csv")

print(f"Total de registros no CSV: {len(df)}")
print(f"Com URL disponível:        {df['url'].notna().sum()}")
print(f"Sem URL (busca manual):    {df['url'].isna().sum()}")
print()

# Registros sem URL — listamos para referência
sem_url = df[df['url'].isna() | (df['url'].str.strip() == '')]
if not sem_url.empty:
    print("📋 Registros sem URL (requerem busca manual):")
    display(sem_url[['instituicao', 'ano', 'tipo_documento', 'status']])

# Trabalhamos apenas com os que têm URL
df_com_url = df[df['url'].notna() & (df['url'].str.strip() != '')].copy()
df_com_url = df_com_url.reset_index(drop=True)

print(f"\n✅ {len(df_com_url)} documentos prontos para download.")

## 3. Criação das pastas de destino

Os arquivos serão organizados em subpastas por **região geográfica**, facilitando a navegação posterior. A estrutura será:

```
documentos_ia_universidades/
├── Norte/
├── Nordeste/
├── Centro-Oeste/
├── Sudeste/
├── Sul/
└── Nacional/
```

In [ ]:
regioes = df_com_url['regiao'].unique()

for regiao in regioes:
    pasta = PASTA_DESTINO / regiao
    pasta.mkdir(parents=True, exist_ok=True)
    print(f"📁 Criada: {pasta}")

print("\n✅ Estrutura de pastas criada.")

## 4. Funções auxiliares

Definimos duas funções:

- **`inferir_extensao(url, response)`** — tenta descobrir se o arquivo é PDF ou HTML a partir da URL e dos headers da resposta HTTP, para nomear o arquivo corretamente.
- **`nome_arquivo(row)`** — gera um nome de arquivo limpo e padronizado a partir dos metadados da linha (instituição + ano + tipo).

In [ ]:
import re

def inferir_extensao(url, response):
    """Determina a extensão do arquivo a partir da URL ou do Content-Type."""
    # Tenta pela URL primeiro
    path = urlparse(url).path.lower()
    if path.endswith(".pdf"):
        return ".pdf"
    if path.endswith(".docx"):
        return ".docx"
    if path.endswith(".doc"):
        return ".doc"
    # Tenta pelo Content-Type da resposta
    content_type = response.headers.get("Content-Type", "")
    if "pdf" in content_type:
        return ".pdf"
    if "word" in content_type or "officedocument" in content_type:
        return ".docx"
    if "html" in content_type:
        return ".html"
    return ".bin"  # fallback genérico


def nome_arquivo(row):
    """Gera nome de arquivo padronizado: INST_ANO_TIPO.ext"""
    inst = re.sub(r"[^\w]", "_", str(row['instituicao']))
    tipo = re.sub(r"[^\w]", "_", str(row['tipo_documento'])[:30])
    return f"{inst}_{row['ano']}_{tipo}"


print("✅ Funções auxiliares definidas.")

## 5. Download dos documentos

Aqui acontece o trabalho principal. Para cada registro com URL:

1. Fazemos a requisição HTTP
2. Verificamos se o servidor respondeu com sucesso (código 200)
3. Checamos se o conteúdo é de fato um arquivo baixável (PDF/DOCX) ou apenas uma página HTML — nesse caso registramos como "revisão manual necessária"
4. Salvamos o arquivo na subpasta da região correspondente
5. Aguardamos um intervalo entre cada download para não sobrecarregar os servidores

Ao final, um log detalhado é exibido.

In [ ]:
log = []  # Registro de resultados

for _, row in tqdm(df_com_url.iterrows(), total=len(df_com_url), desc="Baixando documentos"):
    url = str(row['url']).strip()
    regiao = str(row['regiao'])
    pasta = PASTA_DESTINO / regiao
    base_nome = nome_arquivo(row)

    resultado = {
        "instituicao": row['instituicao'],
        "ano": row['ano'],
        "url": url,
        "status": None,
        "arquivo": None,
        "observacao": ""
    }

    try:
        response = requests.get(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True)

        if response.status_code != 200:
            resultado["status"] = "❌ Erro HTTP"
            resultado["observacao"] = f"Status {response.status_code}"
            log.append(resultado)
            time.sleep(INTERVALO)
            continue

        ext = inferir_extensao(url, response)

        # Páginas HTML: salvar mas marcar para revisão
        if ext == ".html":
            resultado["status"] = "⚠️ Página HTML"
            resultado["observacao"] = "URL aponta para página web, não para arquivo direto. Acesse manualmente."
            log.append(resultado)
            time.sleep(INTERVALO)
            continue

        # Salva o arquivo
        nome_final = base_nome + ext
        caminho = pasta / nome_final

        with open(caminho, "wb") as f:
            f.write(response.content)

        resultado["status"] = "✅ Baixado"
        resultado["arquivo"] = str(caminho)

    except requests.exceptions.Timeout:
        resultado["status"] = "❌ Timeout"
        resultado["observacao"] = "Servidor não respondeu no tempo limite."
    except requests.exceptions.ConnectionError:
        resultado["status"] = "❌ Erro de conexão"
        resultado["observacao"] = "Não foi possível conectar ao servidor."
    except Exception as e:
        resultado["status"] = "❌ Erro inesperado"
        resultado["observacao"] = str(e)

    log.append(resultado)
    time.sleep(INTERVALO)

print("\n✅ Downloads concluídos.")

## 6. Relatório final

Exibimos um resumo dos resultados: quantos foram baixados com sucesso, quantos falharam e quais precisam de atenção manual. O log completo é salvo como CSV para referência futura.

In [ ]:
df_log = pd.DataFrame(log)

# Resumo por status
resumo = df_log['status'].value_counts()
print("=" * 50)
print("RELATÓRIO FINAL")
print("=" * 50)
for status, count in resumo.items():
    print(f"  {status}: {count}")
print("=" * 50)

# Detalhamento dos que precisam de atenção
problemas = df_log[df_log['status'] != "✅ Baixado"]
if not problemas.empty:
    print("\n📋 Documentos que precisam de atenção manual:")
    display(problemas[['instituicao', 'ano', 'status', 'observacao', 'url']])

# Salva o log completo
caminho_log = PASTA_DESTINO / "log_downloads.csv"
df_log.to_csv(caminho_log, index=False, encoding='utf-8-sig')
print(f"\n📄 Log completo salvo em: {caminho_log}")

## 7. Verificação final — arquivos salvos

Lista todos os arquivos baixados, organizados por região, com seus tamanhos.

In [ ]:
print("📂 Arquivos baixados:\n")

total_arquivos = 0
total_bytes = 0

for regiao in sorted(regioes):
    pasta = PASTA_DESTINO / regiao
    arquivos = list(pasta.glob("*"))
    if not arquivos:
        continue
    print(f"📁 {regiao}/")
    for arq in sorted(arquivos):
        tamanho = arq.stat().st_size
        total_bytes += tamanho
        total_arquivos += 1
        print(f"   {arq.name}  ({tamanho/1024:.1f} KB)")
    print()

print(f"Total: {total_arquivos} arquivo(s) — {total_bytes/1024:.1f} KB")